[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aranaur/aranaur.rbind.io/blob/main/lectures/kse/MATH840/26autumn/labs/_demo08.ipynb)

# MATH840 — Week 8 practice: seasonal ARIMA in four real stories

**This session is not marked.** Four cases, each a question someone actually had to answer, worked with
the tools of the *ARIMA Models, Part 2* lecture. Tonight's Lab 7 is the first one, on your own series.

| Case | Story | What it teaches | Minutes |
|---|---|---|---|
| 0 | — | the toolkit, now with seasonal terms | 5 |
| 1 | ✈️ Airline passengers, 1949–1960 | the most famous seasonal ARIMA, read straight off the ACF | 20 |
| 2 | 🌍 CO₂ at Mauna Loa | a model fitted in 2000, checked against 2026 | 20 |
| 3 | 🏭 Six Australian industries | ARIMA against ETS: who wins, and how often | 15 |
| 4 | 🔥 Melbourne's January 2014 heatwave | regression with ARIMA errors, at the edge of its data | 20 |

Every case ends with a question. The answers are the point of the session.

## 0. Setup and the toolkit

In [ ]:
!pip install -q statsforecast==2.1.1

import os
os.environ["NIXTLA_ID_AS_COL"] = "true"
import warnings
warnings.filterwarnings("ignore")
from functools import partial

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.arima import ARIMASummary, ndiffs, nsdiffs
from statsforecast.models import ARIMA, AutoARIMA, AutoETS, SeasonalNaive
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mase

plt.rcParams.update({"figure.figsize": (11, 3.6), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 120, "display.precision", 3)

BASE = ("https://raw.githubusercontent.com/Aranaur/aranaur.rbind.io/"
        "main/lectures/kse/MATH840/26autumn/data")
COLOURS = {"actual": "#314f4f", "a": "#e64173", "b": "#20B2AA", "c": "#FFA500", "d": "#6A5ACD"}

The same four steps as Week 7, with one change: the seasonal lags now get **read**, not just looked at,
and `compare` counts $p + q + P + Q$ for Ljung-Box.

In [ ]:
def difference_plan(y, m):
    # Step 1: D from seasonal strength, then d from KPSS on what is left.
    D = nsdiffs(y, period=m)
    z = y[m:] - y[:-m] if D else y.copy()
    d = ndiffs(z)
    z = np.diff(z, n=d) if d else z
    print(f"D = {D}, d = {d}: {len(y)} observations -> {len(z)} after differencing")
    return D, d, z


def acf_pacf(z, lags, title="", m=None):
    # Step 2: the plots p, q (first lags) and P, Q (lags m, 2m, 3m) are read from.
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
    plot_acf(z, lags=lags, zero=False, ax=axes[0], auto_ylims=True, title=f"{title} ACF")
    plot_pacf(z, lags=lags, zero=False, ax=axes[1], auto_ylims=True, title=f"{title} PACF")
    if m:
        for ax in axes:
            for k in range(m, lags + 1, m):
                ax.axvline(k, color="#e64173", alpha=0.25, linewidth=6)   # the seasonal lags
    plt.show()


def compare(y, candidates):
    # Step 3: AICc and Ljung-Box (2m lags, model_df = p + q + P + Q) for every candidate.
    rows = []
    for name, model in candidates.items():
        fit = model.fit(y=y)
        p, q, P, Q, m = fit.model_["arma"][:5]
        lb = acorr_ljungbox(fit.model_["residuals"], lags=[2 * max(m, 1)], model_df=p + q + P + Q)
        rows.append({"candidate": name, "model": str(ARIMASummary(fit.model_)).strip(),
                     "AICc": fit.model_["aicc"], "Ljung-Box p": lb["lb_pvalue"].iloc[0]})
    return pd.DataFrame(rows).sort_values("AICc")


def rolling(df, models, h, m, windows):
    # Step 4: MASE over `windows` rolling origins, h steps each.
    cv = StatsForecast(models=models, freq=pd.infer_freq(df["ds"])).cross_validation(
        df=df, h=h, n_windows=windows, step_size=h)
    out = evaluate(cv.drop(columns="cutoff"), metrics=[partial(mase, seasonality=m)], train_df=df)
    return out.drop(columns=["unique_id", "metric"]).T.rename(columns={0: "MASE"}).sort_values("MASE")

## 1. ✈️ Airline passengers, 1949–1960

Monthly international airline passengers, in thousands. In 1970 Box and Jenkins used this series to show
the world what an ARIMA model could do; the model they found is still called **the airline model**.

In [ ]:
air = pd.read_csv(f"{BASE}/air_passengers.csv", parse_dates=["ds"]).assign(unique_id="air")
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
axes[0].plot(air["ds"], air["y"], color=COLOURS["actual"])
axes[0].set_title("Passengers: the swing grows with the level")
axes[1].plot(air["ds"], np.log(air["y"]), color=COLOURS["b"])
axes[1].set_title("log(passengers): the swing is constant")
plt.show()

ARIMA has no multiplicative season, so the log does that job (Week 7). Everything below is on
$\log y$.

In [ ]:
log_air = air.assign(y=np.log(air["y"]))
y1 = log_air["y"].to_numpy()
D1, d1, z1 = difference_plan(y1, m=12)
acf_pacf(z1, lags=36, title="log passengers, seasonal + first difference:", m=12)

::: Discussion
**Read the plots before the next cell.** What happens at lag 1? At lag 12 — the shaded seasonal lag?
And at 24 and 36? Write down $(p, d, q)(P, D, Q)_{12}$.
:::

In [ ]:
seasonal = partial(ARIMA, season_length=12)
compare(y1, {
    "by hand: the airline model": seasonal(order=(0, 1, 1), seasonal_order=(0, 1, 1)),
    "by hand: AR instead of MA": seasonal(order=(1, 1, 0), seasonal_order=(1, 1, 0)),
    "AutoARIMA, stepwise": AutoARIMA(season_length=12),
    "AutoARIMA, full search": AutoARIMA(season_length=12, stepwise=False),
})

In [ ]:
airline = seasonal(order=(0, 1, 1), seasonal_order=(0, 1, 1)).fit(y=y1)
print({k: round(float(v), 3) for k, v in airline.model_["coef"].items()},
      " — Box & Jenkins (1970) report -0.40 and -0.61")

resid = airline.model_["residuals"]
fig, axes = plt.subplots(1, 3, figsize=(13, 3), width_ratios=[2, 1, 1])
axes[0].plot(air["ds"], resid, color=COLOURS["c"])
axes[0].set_title("Innovation residuals of the airline model")
plot_acf(resid, lags=36, zero=False, ax=axes[1], auto_ylims=True)
axes[2].hist(resid[13:], bins=20, color=COLOURS["actual"])
axes[2].set_title("Histogram (after the first 13)")
plt.show()

In [ ]:
rolling(log_air,
        [seasonal(order=(0, 1, 1), seasonal_order=(0, 1, 1), alias="airline model"),
         AutoETS(season_length=12, alias="ETS"),
         SeasonalNaive(season_length=12, alias="snaive")],
        h=12, m=12, windows=3)

In [ ]:
fc = airline.predict(h=36, level=[80, 95])
future = pd.date_range(air["ds"].iloc[-1], periods=37, freq="MS")[1:]
fig, ax = plt.subplots()
ax.plot(air["ds"], air["y"], color=COLOURS["actual"], label="passengers")
ax.plot(future, np.exp(fc["mean"]), color=COLOURS["a"], linewidth=2, label="airline model")
for level, alpha in [(95, 0.12), (80, 0.25)]:
    ax.fill_between(future, np.exp(fc[f"lo-{level}"]), np.exp(fc[f"hi-{level}"]),
                    color=COLOURS["a"], alpha=alpha, label=f"{level}%")
ax.set_title("1961–1963, forecast from 1960 — back on the passenger scale")
ax.legend(loc="upper left")
plt.show()

::: Discussion
**Two coefficients and two differences describe twelve years of air travel. The hand reading, the stepwise
search and the full search all land on the same model. When does that happen — and why should it make
you more confident, not less careful?**

And: the interval fans out to the right, and is asymmetric. Which of the model's choices does each of
those come from?
:::

## 2. 🌍 CO₂ at Mauna Loa: a forecast from 2000, checked in 2026

The longest continuous record of atmospheric CO₂, monthly since 1958, in parts per million (NOAA). Here is
the question: **standing in December 2000, with only the data up to then, what would you have forecast
for today — and how wrong would you have been?** Unlike almost every forecast, this one can be checked
over 25 years.

In [ ]:
co2 = pd.read_csv(f"{BASE}/co2_mauna_loa.csv", parse_dates=["ds"])
past, after = co2[co2["ds"] < "2001-01-01"], co2[co2["ds"] >= "2001-01-01"]
y2, y_after = past["y"].to_numpy(), after["y"].to_numpy()
print(f"fit on {len(past)} months to {past['ds'].iloc[-1]:%Y-%m}; check on {len(after)} months to {after['ds'].iloc[-1]:%Y-%m}")
D2, d2, z2 = difference_plan(y2, m=12)
acf_pacf(z2, lags=36, title="CO2 to 2000, seasonal + first difference:", m=12)

In [ ]:
H2 = len(after)
co2_models = {
    "ARIMA by hand (0,1,1)(0,1,1)": seasonal(order=(0, 1, 1), seasonal_order=(0, 1, 1)),
    "AutoARIMA": AutoARIMA(season_length=12),
    "AutoETS": AutoETS(season_length=12),
}
co2_fc, rows = {}, []
for name, model in co2_models.items():
    fit = model.fit(y=y2)
    fc = fit.predict(h=H2, level=[95])
    co2_fc[name] = fc
    error = y_after - fc["mean"]
    rows.append({"model": name,
                 "chose": fit.model_.get("method") or str(ARIMASummary(fit.model_)).strip(),
                 "MAE, 25 years": np.mean(np.abs(error)),
                 f"forecast for {after['ds'].iloc[-1]:%Y-%m}": fc["mean"][-1],
                 "95% interval covered": np.mean((y_after >= fc["lo-95"]) & (y_after <= fc["hi-95"]))})
print(f"actual in {after['ds'].iloc[-1]:%Y-%m}: {y_after[-1]:.1f} ppm")
pd.DataFrame(rows).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(past["ds"], y2, color=COLOURS["actual"], linewidth=0.8, label="known in 2000")
ax.plot(after["ds"], y_after, color="black", linewidth=0.8, label="what happened")
for (name, fc), colour in zip(co2_fc.items(), [COLOURS["a"], COLOURS["d"], COLOURS["b"]]):
    ax.plot(after["ds"], fc["mean"], color=colour, linewidth=1.5, label=name)
    ax.fill_between(after["ds"], fc["lo-95"], fc["hi-95"], color=colour, alpha=0.12)
ax.set_xlim(pd.Timestamp("1985-01-01"), after["ds"].iloc[-1])
ax.set_ylim(340, 440)
ax.set_ylabel("ppm")
ax.legend(loc="upper left", fontsize=8)
ax.set_title("A forecast made in December 2000, with 95% intervals")
plt.show()

Why did every model fall short? Look at the **rate** of growth — the change on the same month a year
earlier, averaged over each year:

In [ ]:
growth = (co2.set_index("ds")["y"].diff(12).resample("YS").mean().dropna())
fig, ax = plt.subplots(figsize=(12, 3.2))
ax.bar(growth.index.year, growth.values, color=[COLOURS["actual"] if y < 2001 else "black" for y in growth.index.year])
ax.axhline(growth[growth.index.year.isin(range(1991, 2001))].mean(), color=COLOURS["a"], linestyle="--",
           label="the 1990s rate the models extrapolated")
ax.set_ylabel("ppm per year")
ax.legend(loc="upper left")
ax.set_title("Annual growth of CO2 at Mauna Loa")
plt.show()

::: Discussion
**The models are not wrong about the past; the world changed under them.** With $d + D = 2$ an ARIMA
forecast is a straight line through the recent slope — the 1990s rate — and emissions accelerated
after 2000.

Two observations to explain: the ARIMA intervals still contained every one of the 309 months, while
ETS's covered fewer than half. What in the two models' structure makes one so much more cautious?

And the one to take home: **what would you tell a policymaker in 2000 about this forecast** — and what
would you have to say differently today?
:::

## 3. 🏭 ARIMA against ETS on six Australian industries

Quarterly production of beer, tobacco, bricks, cement, electricity and gas — six series, one question:
**which family forecasts better?** Six rolling origins, two years ahead each.

In [ ]:
production = pd.read_csv(f"{BASE}/aus_production_long.csv", parse_dates=["ds"])
scores = {}
for industry in ["Beer", "Tobacco", "Bricks", "Cement", "Electricity", "Gas"]:
    df = (production.query("unique_id == @industry").dropna()
          .sort_values("ds").reset_index(drop=True))
    cv = StatsForecast(models=[AutoARIMA(season_length=4, alias="ARIMA"),
                               AutoETS(season_length=4, alias="ETS"),
                               SeasonalNaive(season_length=4, alias="snaive")],
                       freq=pd.infer_freq(df["ds"])).cross_validation(df=df, h=8, n_windows=6, step_size=4)
    cv["ARIMA + ETS"] = (cv["ARIMA"] + cv["ETS"]) / 2
    scores[industry] = (evaluate(cv.drop(columns="cutoff"), metrics=[partial(mase, seasonality=4)], train_df=df)
                        .drop(columns=["unique_id", "metric"]).iloc[0])
scores = pd.DataFrame(scores).T
scores.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.8))
width = 0.2
for j, (column, colour) in enumerate(zip(scores.columns, [COLOURS["d"], COLOURS["b"], COLOURS["a"], COLOURS["c"]])):
    ax.bar(np.arange(len(scores)) + (j - 1.5) * width, scores[column], width, label=column, color=colour)
ax.set_xticks(range(len(scores)), scores.index)
ax.axhline(1, color="grey", linewidth=0.8)
ax.set_ylabel("MASE (lower is better)")
ax.legend(ncol=4)
ax.set_title("Six industries, six rolling origins each")
plt.show()

wins = (scores[["ARIMA", "ETS"]].idxmin(axis=1)).value_counts()
print("wins:", wins.to_dict(), "| the average was never the worst of the three models:",
      bool((scores["ARIMA + ETS"] <= scores[["ARIMA", "ETS"]].max(axis=1)).all()))

::: Discussion
**ARIMA wins more series, ETS wins some, and neither wins all. If you had to forecast a seventh industry
tomorrow, without time to compare, which would you choose — and is "the one that won more often" the
right answer?**

Look at the last column: the average of the two is never the worst. That is the next lecture's main
idea, already visible here.
:::

## 4. 🔥 Melbourne, January 2014: forecasting demand through a heatwave

Between 14 and 17 January 2014 Melbourne had four consecutive days above 41 °C. The grid operator has to
know, days ahead, how much electricity that will take. This is **dynamic regression** (Week 7): daily
demand on temperature, temperature², a working-day flag, and ARIMA errors.

In [ ]:
vic = pd.read_csv(f"{BASE}/vic_elec.csv", parse_dates=["ds"])
wide = vic.pivot_table(index="ds", columns="unique_id", values="y")
wide["holiday"] = vic.query("unique_id == 'Demand'").set_index("ds")["Holiday"]
daily = (wide.groupby(wide.index.date)
         .agg(demand=("Demand", "sum"), temp=("Temperature", "max"), holiday=("holiday", "max"))
         .rename_axis("ds").reset_index())
daily["ds"] = pd.to_datetime(daily["ds"])
daily["demand"] = daily["demand"] / 2000                                  # GWh per day
daily["workday"] = ((daily["ds"].dt.dayofweek < 5) & ~daily["holiday"].astype(bool)).astype(int)

origin = int(np.flatnonzero(daily["ds"] == "2014-01-06")[0])              # forecast made on 6 January
H4 = 21
train4 = daily.iloc[:origin]
window = daily.iloc[origin:origin + H4]
print(f"training: {train4['ds'].iloc[0]:%Y-%m-%d} to {train4['ds'].iloc[-1]:%Y-%m-%d};  "
      f"hottest day in training: {train4['temp'].max():.1f} °C;  hottest in the window: {window['temp'].max():.1f} °C")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(train4["temp"], train4["demand"], s=8, color="lightgray", label="2012–2013")
heat = window[window["temp"] > 40]
ax.scatter(heat["temp"], heat["demand"], s=60, color=COLOURS["a"], label="the heatwave days")
ax.set_xlabel("maximum temperature, °C")
ax.set_ylabel("GWh per day")
ax.legend()
ax.set_title("The heatwave sits at the very edge of everything the model has seen")
plt.show()

In [ ]:
X = np.column_stack([daily["temp"], daily["temp"] ** 2, daily["workday"]])
y4 = daily["demand"].to_numpy()

with_temp = AutoARIMA(season_length=7).fit(y=y4[:origin], X=X[:origin])
fc_temp = with_temp.predict(h=H4, X=X[origin:origin + H4], level=[80])
fc_none = AutoARIMA(season_length=7).fit(y=y4[:origin]).predict(h=H4, level=[80])

actual = y4[origin:origin + H4]
print(ARIMASummary(with_temp.model_))
print(f"MAE over the 21 days:  with temperature {np.mean(np.abs(actual - fc_temp['mean'])):.1f} GWh,  "
      f"without {np.mean(np.abs(actual - fc_none['mean'])):.1f} GWh")
heatwave = (window["temp"] > 40).to_numpy()
print(f"on the heatwave days:  actual {actual[heatwave].mean():.0f},  with temperature "
      f"{fc_temp['mean'][heatwave].mean():.0f},  without {fc_none['mean'][heatwave].mean():.0f} GWh per day")

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 4))
days = daily["ds"].iloc[origin - 14:origin + H4]
ax1.plot(days, y4[origin - 14:origin + H4], color=COLOURS["actual"], linewidth=1.5, label="actual demand")
ax1.plot(window["ds"], fc_temp["mean"], color=COLOURS["a"], linewidth=2, label="with temperature")
ax1.fill_between(window["ds"], fc_temp["lo-80"], fc_temp["hi-80"], color=COLOURS["a"], alpha=0.15)
ax1.plot(window["ds"], fc_none["mean"], "--", color=COLOURS["d"], linewidth=1.8, label="without temperature")
ax1.axvline(window["ds"].iloc[0], color="grey", linewidth=0.8)
ax1.set_ylabel("GWh per day")
ax2 = ax1.twinx()
ax2.bar(days, daily["temp"].iloc[origin - 14:origin + H4], color=COLOURS["c"], alpha=0.2, width=0.8)
ax2.set_ylabel("max temperature, °C", color=COLOURS["c"])
ax2.grid(False)
ax1.legend(loc="upper left")
ax1.set_title("Forecasts made on 6 January 2014 — with the temperature that actually came")
plt.show()

Look past the heatwave too: from 10 to 24 January the actual demand sits **above the 80% interval fifteen
days in a row** — including the week after, when the temperature was back to 22–24 °C.

In practice the operator does not know the temperature either — only a weather forecast. So the useful
output is a **set of scenarios**: the same model, fed three versions of the next three weeks.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.8))
ax.plot(window["ds"], actual, color=COLOURS["actual"], linewidth=1.5, label="what happened")
for shift, colour, label in [(-5, COLOURS["b"], "5 °C cooler than it was"), (0, COLOURS["a"], "as it was"),
                             (5, COLOURS["c"], "5 °C hotter than it was")]:
    t = window["temp"].to_numpy() + shift
    Xs = np.column_stack([t, t ** 2, window["workday"]])
    ax.plot(window["ds"], with_temp.predict(h=H4, X=Xs)["mean"], color=colour, linewidth=1.8, label=label)
ax.set_ylabel("GWh per day")
ax.legend(loc="upper left")
ax.set_title("Scenario forecasts: the same model, three weather assumptions")
plt.show()

::: Discussion
**Temperature closes about half of the gap on the heatwave days, and not more. Why?** Two reasons worth
finding: the heatwave is hotter than any day in the training data, so the quadratic curve is
extrapolating; and air conditioners, once every one of them is on, do not respond to temperature the
way a parabola does.

Look at the scenario plot once more: the line closest to what actually happened is the orange one — the
weather made **5 °C hotter than it really was** — and even it falls short on the peak days. The model's
response to heat is too gentle at the top.

And the operator's question: is the forecast **with** temperature good enough to plan with, when its 80%
interval missed 15 of the 21 days? What would you add to the model before the next summer?
:::

## What carries to tonight

| Case | The move | What decided |
|---|---|---|
| ✈️ Airline | log, $D = 1$, $d = 1$; spikes at 1 and 12 → (0,1,1)(0,1,1) | hand, stepwise and full search agreed; rolling origin confirmed |
| 🌍 CO₂ | the same structure, 25 years ahead | the world changed: honest intervals mattered more than the point |
| 🏭 Industries | ARIMA vs ETS on six series | neither wins everywhere; the average never lost |
| 🔥 Heatwave | regression with ARIMA errors | a predictor helps only as far as its training range |

**Lab 7** is case 1 on your own series: difference, read $p, q$ **and** $P, Q$ from the ACF and PACF,
compare with `AutoARIMA` on AICc and a rolling origin, check residuals with $p + q + P + Q$, submit.